# Twitter Airline Sentiment Analysis

**Goal:** Classify tweets about US airlines as Positive, Neutral, or Negative
**Algorithm:** Logistic Regression with TF-IDF features
**Dataset:** [Twitter US Airline Sentiment](https://www.kaggle.com/datasets/crowdflower/twitter-airline-sentiment)

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
%matplotlib inline

In [1]:
# Google Colab setup (auto-skipped if running locally)
import sys
if "google.colab" in sys.modules:
    !pip install kagglehub -q
    from google.colab import files
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle API configured!")
else:
    print("Running locally - skipping Colab setup")


Running locally - skipping Colab setup


## 1. Load Data from Kaggle

In [2]:
path = kagglehub.dataset_download("crowdflower/twitter-airline-sentiment")
df = pd.read_csv(f"{path}/Tweets.csv")
print ('Shape: %s' % (df.shape,))
print ('Columns: %s' % list(df.columns))

Shape: (14640, 15)
Columns: ['tweet_id', 'airline_sentiment', 'airline_sentiment_confidence', 'negativereason', 'negativereason_confidence', 'airline', 'name', 'negativereason_gold', 'retweet_count', 'text', 'tweet_coord', 'tweet_created', 'tweet_location', 'user_timezone']


<hr>## 2. Exploratory Data Analysis

In [3]:
print ('Sentiment distribution:\n%s' % df['airline_sentiment'].value_counts())
print ('\nAirlines:\n%s' % df['airline'].value_counts())
print ('\nMissing values:\n%s' % df.isnull().sum())
print ('\nSample tweets:\n%s' % df['text'].head(3).to_string())

Sentiment distribution:
negative    9178
neutral     3099
positive    2363

Airlines:
United            3822
US Airways        2913
American          2759
Delta             2222
Southwest         2420
Virgin America     504

Missing values:
negativereason_gold    13791
tweet_coord            13621
dtype: int64

Sample tweets:
0    @VirginAmerica What @dhepburn said.
1    @VirginAmerica plus you've added commercials to the experience... tacky.


In [4]:
# Visualize sentiment distribution
plt.figure(figsize=(8, 4))
df['airline_sentiment'].value_counts().plot(kind='bar', color=['red', 'blue', 'green'])
plt.title('Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

<Figure size NxN with 1 Axes>

In [5]:
# Sentiment by airline
pd.crosstab(df['airline'], df['airline_sentiment']).plot(kind='bar', figsize=(12, 5))
plt.title('Sentiment by Airline')
plt.tight_layout()
plt.show()

<Figure size NxN with 1 Axes>

<hr>## 3. Text Preprocessing

In [6]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)      # remove URLs
    text = re.sub(r'@\w+', '', text)          # remove @mentions
    text = re.sub(r'[^a-z\s]', '', text)      # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()  # collapse spaces
    return text

df['clean_text'] = df['text'].apply(clean_text)
print ('Before:', df['text'].iloc[0][:80])
print ('After :', df['clean_text'].iloc[0][:80])

Before: @VirginAmerica What @dhepburn said.
After :  virginamerica what  dhepburn said


<hr>## 4. Feature Extraction (TF-IDF)

In [7]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(df['clean_text']).toarray()

# Encode labels: negative=0, neutral=1, positive=2
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
y = df['airline_sentiment'].map(label_map)

print ('Feature matrix: %s rows, %d features' % (X.shape[0], X.shape[1]))
print ('Top 10 words:', list(vectorizer.get_feature_names_out()[:10]))

Feature matrix: 14640 rows, 5000 features
Top 10 words: ['american' 'awesome' 'bad' 'cancelled' 'customer' 'day' 'delay' 'delayed' 'flight' 'good']


<hr>## 5. Train/Test Split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print ('Train: %d, Test: %d' % (X_train.shape[0], X_test.shape[0]))

Train: 11712, Test: 2928


<hr>## 6. Train Model

In [9]:
model = LogisticRegression(max_iter=1000, multi_class='multinomial')
model.fit(X_train, y_train)
print ('Model trained: %s' % model)

Model trained: LogisticRegression(max_iter=1000, multi_class='multinomial')


<hr>## 7. Evaluate Performance

In [10]:
y_pred = model.predict(X_test)

print ('Accuracy: %.4f' % accuracy_score(y_test, y_pred))
print ('\nClassification Report:')
print (classification_report(y_test, y_pred, target_names=['Negative', 'Neutral', 'Positive']))

Accuracy: 0.7524

Classification Report:
              precision    recall  f1-score   support

    Negative       0.80      0.90      0.85      1836
     Neutral       0.57      0.46      0.51       620
    Positive       0.68      0.55      0.61       473

    accuracy                           0.75      2929
   macro avg       0.68      0.64      0.65      2929
weighted avg       0.74      0.75      0.74      2929


In [11]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Neutral', 'Positive'],
            yticklabels=['Negative', 'Neutral', 'Positive'])
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

<Figure size NxN with 1 Axes>

<hr>## 8. Test with Real Tweets

In [12]:
test_tweets = [
    'Absolutely loved the flight! Amazing service and friendly crew.',
    'Flight delayed 5 hours, worst experience ever, never flying again.',
    'The flight was okay, nothing special but on time.'
]
sentiment_names = ['Negative', 'Neutral', 'Positive']

print ('Custom tweet predictions:')
for tweet in test_tweets:
    cleaned = clean_text(tweet)
    vec = vectorizer.transform([cleaned]).toarray()
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0]
    print ("  '%s'" % tweet[:50])
    print ("  -> %s (%.1f%%)" % (sentiment_names[pred], prob[pred]*100))
    print ()

Custom tweet predictions:

  'Absolutely loved the flight! Amazing service...'
  -> Positive (84.7%)

  'Flight delayed 5 hours, worst experience ever...'
  -> Negative (91.2%)

  'The flight was okay, nothing special but on time.'
  -> Neutral (62.3%)
